# Tutorial 2 — Agent Enhancement: Tool Use, Multi-Turn, Opik

Tutorial 1 left you at `backend.complete(system, user)`. That is enough for one-shot prompting, but agents add three things on top:

1. **Tool use** — the LLM picks among `@function_tool` definitions and SEOCHO actually runs them. Eight tools ship in `seocho/tools.py` (5 indexing, 2–3 query).
2. **Multi-turn design** — `Session` accumulates entities and relationships across `add()` calls so later `ask()` calls have context. State is structured (entity cache + query cache), not raw chat history.
3. **Observability with Opik** — every `Session.add` / `ask` / `run` logs a span; Opik visualizes them; toggle on/off via env var.

Three execution modes climb that ladder:

| Mode | Entry point | What runs |
|------|-------------|-----------|
| `pipeline` (default) | `Session.add` / `Session.ask` | Deterministic chunk → extract → validate → link → write. No LLM reasoning about flow. |
| `agent` | `Session.add` / `Session.ask` | LLM agent calls the tools. Falls back to pipeline on error. |
| `supervisor` | `Session.run` | Router agent hands off to IndexingAgent or QueryAgent. |

Source references — `seocho/agent_config.py`, `seocho/session.py`, `seocho/agent/factory.py`, `seocho/tools.py`, `seocho/tracing.py`.

## 0. Prerequisites

In [ ]:
# --- Setup (run me first) ---
import sys
from pathlib import Path

try:
    import seocho
    from dotenv import load_dotenv
except ImportError:
    %pip install -q "seocho[local]" python-dotenv
    import seocho
    from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env", override=False)
print('seocho', seocho.__version__, '| .env loaded')


In [6]:
import os, getpass

PROVIDERS = [
    ('openai',   'OPENAI_API_KEY',     'gpt-4o-mini'),
    ('grok',     'XAI_API_KEY',        'grok-4.20-reasoning'),
    ('kimi',     'MOONSHOT_API_KEY',   'kimi-k2.5'),
    ('deepseek', 'DEEPSEEK_API_KEY',   'deepseek-chat'),
]

# Prompt for any keys not already set (.env / shell / earlier cells).
# Press Enter to skip a provider you don't have a key for.
for _, env_name, _ in PROVIDERS:
    if not os.environ.get(env_name):
        val = getpass.getpass(f'{env_name} (blank to skip): ')
        if val:
            os.environ[env_name] = val

AVAILABLE = [(p, env, m) for p, env, m in PROVIDERS if os.environ.get(env)]
LIVE = bool(AVAILABLE)

if LIVE:
    PROVIDER, _, MODEL = AVAILABLE[0]
    LLM_SPEC = f'{PROVIDER}/{MODEL}'
    print(f'live cells will run via {LLM_SPEC}')
else:
    PROVIDER = MODEL = LLM_SPEC = None
    print('[NOTE] no provider key set — live cells will skip.')

live cells will run via openai/gpt-4o-mini


## 1. A small ontology — placeholder for the agent shell

`Session` requires an ontology to construct the indexing/query agents (the tools need to know what entity types exist). For this tutorial we use a hand-rolled Person/Company schema; Tutorial 3 swaps in FIBO and shows the ontology *driving* extraction quality.

In [7]:
from seocho import Ontology, NodeDef, RelDef, P

ontology = Ontology(
    name='finance_demo',
    nodes={
        'Person':  NodeDef(properties={'name': P(str, unique=True), 'title': P(str)}),
        'Company': NodeDef(properties={'name': P(str, unique=True), 'sector': P(str)}),
    },
    relationships={
        'WORKS_AT': RelDef(source='Person', target='Company', cardinality='MANY_TO_ONE'),
    },
)
print('validate :', ontology.validate() or 'OK')

validate : OK


## 2. Pillar 1 — Tool use

Pipeline mode runs a deterministic pipeline; agent mode lets the LLM **decide** when to call which tool. The tools are real Python functions decorated with `@function_tool`. The LLM sees their signatures and docstrings, picks the right ones, and SEOCHO executes them.

**Indexing tools** (`create_indexing_tools()` in `seocho/tools.py`):

1. `extract_entities(text, category)` — LLM extraction against the ontology.
2. `validate_extraction(extraction_json)` — SHACL validation.
3. `score_extraction(extraction_json)` — quality score.
4. `link_entities(extraction_json)` — dedup by `label::name`.
5. `write_to_graph(extraction_json, database, source_id)` — persist.

**Query tools** (`create_query_tools()`):

6. `text2cypher(intent, anchor_entity, ...)` — intent-driven Cypher build.
7. `execute_cypher(cypher, params_json, database)` — read against the graph.
8. `search_similar(query, limit)` — optional, only with a vector store wired.

In [ ]:
from seocho import Seocho, AGENT_PRESETS

TEXT = 'Tim Cook is the CEO of Apple, a technology company.'

if LIVE:
    # Pipeline mode — no tool calls, deterministic pipeline
    s_pipeline = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae')
    with s_pipeline.session('pipeline-demo') as sess:
        out = sess.add(TEXT)
        print('[pipeline ]', f"mode={out.get('mode')}  nodes={out.get('nodes_created')}  rels={out.get('relationships_created')}")
    s_pipeline.close()

    # Agent mode — LLM picks among the 5 indexing tools
    s_agent = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae', agent_config=AGENT_PRESETS['agent'])
    with s_agent.session('agent-demo') as sess:
        out = sess.add(TEXT)
        print('[agent    ]', f"mode={out.get('mode')}  nodes={out.get('nodes_created')}  rels={out.get('relationships_created')}  degraded={out.get('degraded', False)}")
    s_agent.close()
else:
    print('[SKIP] tool-use demo — set OPENAI_API_KEY to run.')

If the agent path raises (tool failure, network blip, Agents SDK not installed), `Session._add_via_agent` catches it, calls `_add_via_pipeline` instead, and stamps `degraded=True`, `fallback_from='agent'`, `fallback_reason=<exc>` on the result. Read these fields to know whether your run actually used the agent.

## 3. Pillar 2 — Multi-turn design

A `Session` is not a chat history. It is a structured cache of (a) extracted entities by source, (b) extracted relationships, and (c) recent answers. When you call `session.ask(...)`, the prior `add()` calls have already populated this cache, and the agent receives it as context — without you re-shipping any text.

Three things to watch:

- The **entity cache** grows with every `add()`. The agent can answer about subjects from earlier turns.
- The **query cache** memoizes recent answers — repeating the same `ask()` returns instantly.
- The **session id and database** are propagated to every span so traces (next pillar) are grouped automatically.

In [ ]:
if LIVE:
    s = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae')
    with s.session('multi-turn-demo') as sess:
        # Turn 1 — establish facts
        sess.add('Tim Cook is the CEO of Apple, a technology company.')
        # Turn 2 — extend
        sess.add('Jensen Huang leads NVIDIA, an AI hardware company.')
        # Turn 3 — refer back
        sess.add('Apple sells iPhones; NVIDIA sells GPUs.')

        # Inspect the cache the next ask() will see
        ctx = sess.context.to_agent_context(ontology=ontology)
        print('--- structured context the agent receives at ask() ---')
        print(ctx[:400] + ('...' if len(ctx) > 400 else ''))
        print()

        # Multi-turn answer — references content from turn 1 + 2
        print('answer (cross-turn):', sess.ask('Compare the CEOs of Apple and NVIDIA.'))

        # Cache hit — repeating the question returns instantly
        print('answer (cache hit):', sess.ask('Compare the CEOs of Apple and NVIDIA.'))
    s.close()
else:
    print('[SKIP] multi-turn demo — set OPENAI_API_KEY to run.')

Inspecting `sess.context.history` and `sess.context.entity_index` after a session reveals what got cached. The relevant fields:

- `add_indexing(source_id, nodes, rels, content, mode, degraded, ...)` — recorded after each `add()`.
- `add_query(question, answer, mode, degraded, ...)` — recorded after each `ask()` (cache adds use `mode='cache'`).
- `register_entities(extracted_nodes, source_id, db)` — populates the entity cache the agent reads on the next turn.

## 4. Pillar 3 — Observability with Opik

Up to now everything is invisible — you see one answer per call, but not the path it took. SEOCHO ships a vendor-neutral tracing layer (`seocho/tracing.py`) with four backends — `none`, `console`, `jsonl`, `opik` — and you can run several at once.

The pattern:

1. **Off by default.** No tracing until you call `enable_tracing(...)` or set `SEOCHO_TRACE_BACKEND`.
2. **JSONL is the canonical artifact.** Portable, grep-able, commit-friendly under `traces/`.
3. **Opik is the optional team exporter.** Set `OPIK_API_KEY` (or configure `~/.opik.config`) and add `"opik"` to the backend list.
4. **`disable_tracing()`** turns everything off again.

Every `Session.add` / `ask` / `run` call logs a span carrying the active `agent_config` (including `routing_policy` weights), so policy effects become visible in the trace stream.

In [ ]:
from seocho.tracing import (
    enable_tracing, disable_tracing,
    is_tracing_enabled, is_backend_enabled, current_backend_names,
    flush_tracing,
)

# On/off mode, in priority order:
#   TUTORIAL_TRACE='off'   → no tracing
#   TUTORIAL_TRACE='opik'  → Opik + JSONL (requires OPIK_API_KEY)
#   TUTORIAL_TRACE='jsonl' → JSONL only
#   anything else (auto)   → Opik+JSONL if OPIK_API_KEY is set, else JSONL
TUTORIAL_TRACE = os.environ.get('TUTORIAL_TRACE', 'auto').lower()
JSONL_PATH = './traces/tutorial_02.jsonl'

if TUTORIAL_TRACE == 'off':
    disable_tracing()
elif TUTORIAL_TRACE == 'opik' or (TUTORIAL_TRACE == 'auto' and os.environ.get('OPIK_API_KEY')):
    enable_tracing(
        backend=['opik', 'jsonl'],
        output=JSONL_PATH,
        project_name='seocho-openai',
    )
else:
    enable_tracing(backend='jsonl', output=JSONL_PATH)

print('tracing enabled  :', is_tracing_enabled())
print('active backends  :', current_backend_names() or '(none)')
print('opik backend on? :', is_backend_enabled('opik'))
print('JSONL artifact   :', JSONL_PATH)

## 5. RoutingPolicy — a trace-side concern

`RoutingPolicy(latency, token_efficiency, information_quality)` weights are read by the supervisor's system prompt to decide retries, validation strictness, and answer style. The effect is invisible from a single answer; it shows up in traces as differences in span count, retry frequency, and elapsed time.

Three named presets:

- `RoutingPolicy.fast()` — `latency=0.7`.
- `RoutingPolicy.balanced()` — equal weights.
- `RoutingPolicy.thorough()` — `information_quality=0.8`, retries enabled.

In [ ]:
from seocho import RoutingPolicy, AgentConfig

for label, policy in [
    ('fast    ', RoutingPolicy.fast()),
    ('balanced', RoutingPolicy.balanced()),
    ('thorough', RoutingPolicy.thorough()),
]:
    h = policy.to_agent_hints()
    print(f'{label}  dominant={policy.dominant_axis:<22}  '
          f'qual_threshold={h["extraction_quality_threshold"]:.2f}  '
          f'retries={h["extraction_max_retries"]}  '
          f'repair={h["repair_budget"]}  '
          f'validation={h["validation_on_fail"]:<6}  '
          f'answer={h["answer_style"]}')

Run the supervisor under two contrasting policies — the same call, different policy, distinct trace shapes. With Opik enabled you see two traces in the project dashboard; with JSONL only, `tail -f traces/tutorial_02.jsonl | jq` shows the spans live.

In [ ]:
if LIVE:
    for label, policy in [
        ('fast    ', RoutingPolicy.fast()),
        ('thorough', RoutingPolicy.thorough()),
    ]:
        cfg = AgentConfig(execution_mode='supervisor', handoff=True, routing_policy=policy)
        s = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae', agent_config=cfg)
        with s.session(f'trace-{label.strip()}') as sess:
            sess.run('Index this fact: Sundar Pichai is the CEO of Google.')
            answer = sess.run('Who is the CEO of Google?')
        s.close()
        print(f'  [{label}] policy={policy.dominant_axis:<22} answer_preview={answer[:80]!r}')
    flush_tracing()
    print()
    if is_backend_enabled('opik'):
        print('Spans sent to Opik project ' + repr('seocho-openai') + ' and JSONL at ' + JSONL_PATH)
    else:
        print('Spans written to JSONL at ' + JSONL_PATH + ' (set OPIK_API_KEY to also send to Opik).')
else:
    print('[SKIP] tracing demo — set OPENAI_API_KEY to run.')

**Turn it off when you are done.** Tracing has overhead; Opik writes go over the network. Leaving it enabled in interactive sessions is a foot-gun.

In [ ]:
disable_tracing()
print('tracing enabled  :', is_tracing_enabled())
print('active backends  :', current_backend_names() or '(none)')

## 6. Composition patterns — sequential, parallel, supervisor

The three pillars (tool use / multi-turn / observability) cover a single agent run. Real workflows compose multiple runs. Three patterns ship out of the box — each is a one-line API change:

| Pattern | API | When to use |
|---------|-----|-------------|
| **Sequential** | repeated `sess.ask(...)` | turn 2 needs turn 1's answer |
| **Parallel** | `asyncio.gather` + `asyncio.to_thread(sess.ask, ...)` | independent fan-out, latency-bound |
| **Supervisor** | `AGENT_PRESETS['supervisor']` + `sess.run(...)` | NL message of ambiguous index/query intent |

All six examples below reuse `ontology` and the `LLM_SPEC` already configured. Each pattern shows a **basic** form first, then an **advanced** form.

In [ ]:
# Shared client used across the compose patterns. We re-create it here so the
# section is self-contained even if you skip earlier cells.
if LIVE:
    s_patterns = Seocho.local(
        ontology,
        llm=LLM_SPEC,
        user_id='openai-yitae',     # stamps every span
    )
    # Seed the graph with a couple of facts so the queries below have something to chew on.
    with s_patterns.session('seed') as sess:
        sess.add('Tim Cook is the CEO of Apple, a technology company.')
        sess.add('Jensen Huang is the CEO of NVIDIA, an AI hardware company.')
        sess.add('Sundar Pichai is the CEO of Google.')
    print('seeded Apple/NVIDIA/Google facts under workspace=default, user=openai-yitae')
else:
    s_patterns = None
    print('[SKIP] compose patterns — set a provider key.')

### 6.1 Sequential — basic (chain `ask` → `ask`)

The simplest composition: take the answer from one `ask()`, feed it to the next. The session's entity cache means turn 2 already knows the entities turn 1 surfaced — no re-shipping required.

In [ ]:
if LIVE and s_patterns is not None:
    with s_patterns.session('seq-basic') as sess:
        ceo = sess.ask('Who is the CEO of Apple?')
        about = sess.ask(f'What other facts do we know about {ceo!r}?')

        print('Step 1 — CEO:', ceo)
        print('Step 2 — Profile:', about[:200] + ('...' if len(about) > 200 else ''))
else:
    print('[SKIP]')

### 6.2 Sequential — advanced (extract → relate → synthesize)

A three-step chain: enumerate entities, query their relationships, ask the LLM for a synthesis. Each step grounds the next; the synthesis is the LLM's job, not the graph's.

In [ ]:
if LIVE and s_patterns is not None:
    with s_patterns.session('seq-advanced') as sess:
        entities      = sess.ask('List every Person and Company node, one per line.')
        relationships = sess.ask('For each Person above, what Company do they WORK_AT?')
        narrative     = sess.ask('Synthesize the Person/Company landscape we discovered into 2-3 sentences.')

        print('--- step 1: entities ---');     print(entities[:300])
        print('\n--- step 2: relationships ---'); print(relationships[:300])
        print('\n--- step 3: narrative ---');     print(narrative[:400])
else:
    print('[SKIP]')

### 6.3 Parallel — basic (`asyncio.gather` over independent asks)

When the questions don't depend on each other, fan out and gather. Two important details:

1. **Each branch gets its own session.** `Session` is not thread-safe (the entity_index and query_cache are shared dicts) — fanning into one session would race.
2. **`asyncio.to_thread(sess.ask, q)`** keeps the event loop free while the synchronous `ask` runs in a worker thread. seocho 0.3.2's `_run_sync` makes this Jupyter-safe automatically.

Latency wins: 3 questions in roughly the time of the slowest one.

In [ ]:
import asyncio, time

async def parallel_ask(client, questions):
    async def one(q):
        # Each branch isolates its own session — see note above.
        with client.session(f'par-{abs(hash(q)) & 0xfff:03x}') as sess:
            return await asyncio.to_thread(sess.ask, q)
    return await asyncio.gather(*[one(q) for q in questions])

if LIVE and s_patterns is not None:
    questions = [
        'Who is the CEO of Apple?',
        'Who is the CEO of NVIDIA?',
        'Who is the CEO of Google?',
    ]

    t0 = time.time()
    answers = asyncio.run(parallel_ask(s_patterns, questions))
    elapsed = time.time() - t0

    for q, a in zip(questions, answers):
        print(f'{q[:40]!r:<45} → {a[:80]}')
    print(f'\nElapsed: {elapsed:.2f}s for {len(questions)} parallel queries '
          f'(vs. ~{elapsed*len(questions):.1f}s sequential)')
else:
    print('[SKIP]')

### 6.4 Parallel — advanced (vote across `RoutingPolicy` settings)

Same question, three different `RoutingPolicy` settings, in parallel. Each policy gives a different answer shape (concise vs. evidence-cited); a quorum across them produces a more robust answer than any single policy. **Cost: 3× tokens. When to use: high-stakes single-shot questions where you can afford the spend.**

In [ ]:
from collections import Counter

async def vote_across_policies(question):
    async def one(label, policy):
        cfg = AgentConfig(execution_mode='supervisor', handoff=True, routing_policy=policy)
        s_var = Seocho.local(ontology, llm=LLM_SPEC, agent_config=cfg, user_id='openai-yitae')
        try:
            with s_var.session(f'vote-{label.strip()}') as sess:
                return label, await asyncio.to_thread(sess.run, question)
        finally:
            s_var.close()
    return await asyncio.gather(
        one('fast    ', RoutingPolicy.fast()),
        one('balanced', RoutingPolicy.balanced()),
        one('thorough', RoutingPolicy.thorough()),
    )

if LIVE and s_patterns is not None:
    variants = asyncio.run(vote_across_policies('Who is the CEO of NVIDIA?'))
    for label, ans in variants:
        print(f'  [{label}] {ans[:100]}')

    # Crude quorum: split on first sentence boundary, take majority
    short = [ans.split('.')[0].strip() for _, ans in variants]
    quorum = Counter(short).most_common(1)[0][0]
    print(f'\nQuorum answer: {quorum!r}')
else:
    print('[SKIP]')

### 6.5 Supervisor — basic (auto-route between IndexingAgent and QueryAgent)

`AGENT_PRESETS['supervisor']` enables `execution_mode='supervisor'` + `handoff=True`. Now `sess.run(message)` accepts any natural-language message; the supervisor dispatches to **IndexingAgent** for new facts or **QueryAgent** for questions. **No verb required from the caller.**

In [ ]:
if LIVE and s_patterns is not None:
    s_sup = Seocho.local(
        ontology, llm=LLM_SPEC,
        agent_config=AGENT_PRESETS['supervisor'],
        user_id='openai-yitae',
    )
    try:
        with s_sup.session('sup-basic') as sess:
            # Supervisor sees an indexing intent — routes to IndexingAgent
            sess.run('Add this: Lisa Su is the CEO of AMD.')
            # Supervisor sees a query intent — routes to QueryAgent
            answer = sess.run('Who is the CEO of AMD?')
            print('Supervisor answer:', answer)
    finally:
        s_sup.close()
else:
    print('[SKIP]')

### 6.6 Supervisor — advanced (policy trade-off, visible in traces)

`supervisor_fast` and `supervisor_thorough` differ only in `routing_policy` — **same code, different trace shape**. With JSONL tracing on, you can compare span counts, retry frequency, and latency directly.

Run this with tracing enabled and `tail -f traces/tutorial_02_patterns.jsonl | jq '.metadata.elapsed_seconds'` to see the cost difference live.

In [ ]:
from seocho.tracing import enable_tracing, flush_tracing, disable_tracing

if LIVE and s_patterns is not None:
    enable_tracing(backend='jsonl', output='./traces/tutorial_02_patterns.jsonl')

    QUESTION = 'Compare the CEOs of Apple and NVIDIA, including any other facts about them.'

    for label, preset_name in [('fast    ', 'supervisor_fast'),
                                ('thorough', 'supervisor_thorough')]:
        s_sup = Seocho.local(
            ontology, llm=LLM_SPEC,
            agent_config=AGENT_PRESETS[preset_name],
            user_id='openai-yitae',
        )
        try:
            with s_sup.session(f'sup-{label.strip()}') as sess:
                t0 = time.time()
                answer = sess.run(QUESTION)
                elapsed = time.time() - t0
            print(f'  [{label}] {elapsed:.2f}s → {answer[:120]}')
        finally:
            s_sup.close()

    flush_tracing()
    disable_tracing()
    print('\nSpans appended to ./traces/tutorial_02_patterns.jsonl — '
          'compare elapsed_seconds and span counts per policy.')
else:
    print('[SKIP]')

### 6.7 Pattern comparison — when to pick which

| Pattern | Tokens | Latency | Robustness | Use when... |
|---------|-------:|--------:|------------|-------------|
| Sequential basic | 1× | 1× | low | turn 2 needs turn 1's answer verbatim |
| Sequential advanced | 3× | 3× | medium | building a multi-step argument |
| Parallel basic | N× | ~1× | low (per branch) | N independent questions, latency-bound |
| Parallel advanced (vote) | 3× | ~1× | high | one critical question, want quorum |
| Supervisor basic | 1× | 1.2× | medium | NL message of ambiguous index/query intent |
| Supervisor advanced | 1× | 0.8–2× | tunable | want to dial the latency/quality slider |

**Common pitfall:** the parallel pattern wants its own `Session` per branch (Session is not thread-safe). The advanced parallel-vote pattern goes one step further — its own `Seocho` per branch, so each branch carries its own `agent_config`.

All six patterns emit the same span-metadata contract (`user_id`, `workspace_id`, `degraded` if fallback fired, `fallback_from`). Filter Opik on `metadata.user_id == 'openai-yitae'` to see exactly the runs from this section.

### 6.8 Adaptive escalation — basic (try `fast()`, escalate to `thorough()` on degraded)

The first six patterns picked a policy *up front*. The closed-loop pattern lets the **result tell you** which policy to use:

1. Try `RoutingPolicy.fast()`.
2. Inspect the result — `degraded=True`, empty answer, or `fallback_from='agent'` are all signals.
3. Only on a signal, retry with `RoutingPolicy.thorough()`.

This is **the cheapest correct policy**: you spend tokens proportional to difficulty, not the worst-case difficulty. Both spans land in Opik with their `routing_policy` recorded — easy to filter for the escalation rate over time.

In [ ]:
def adaptive_ask(client, question, *, fast_first=True):
    """Try fast(); only retry under thorough() on a degraded signal.

    Returns (answer, policy_used, escalated_bool).
    """
    policies = [RoutingPolicy.fast(), RoutingPolicy.thorough()] if fast_first else [RoutingPolicy.thorough()]
    last_answer = ''
    for i, policy in enumerate(policies):
        cfg = AgentConfig(execution_mode='supervisor', handoff=True, routing_policy=policy)
        s_local = Seocho.local(ontology, llm=LLM_SPEC, agent_config=cfg, user_id='openai-yitae')
        try:
            with s_local.session(f'escalate-{policy.dominant_axis}') as sess:
                last_answer = sess.run(question)
                # Degraded signal: empty answer is the cheapest one to read after .run()
                # (.add()/.ask() also expose result.get('degraded'); here .run() returns text only)
                degraded = (not last_answer) or last_answer.strip() in ('', 'No response from agent.')
        finally:
            s_local.close()
        if not degraded:
            return last_answer, policy.dominant_axis, i > 0
    return last_answer, policies[-1].dominant_axis, len(policies) > 1

if LIVE and s_patterns is not None:
    answer, policy_used, escalated = adaptive_ask(s_patterns,
        'List every CEO in the graph and the company they lead.')
    print(f'policy used  : {policy_used}')
    print(f'escalated    : {escalated}')
    print(f'answer       : {answer[:200]}')
else:
    print('[SKIP]')

### 6.9 Trace-driven policy selection — advanced (Opik traces feed the next policy)

With JSONL tracing on, you have a record of every span — `routing_policy`, `elapsed_seconds`, `degraded`, `fallback_from`. **The next policy you pick can be a function of those metrics.**

Pattern: run a batch of representative questions across all three policies, score each policy by a fitness function (e.g., `mean(elapsed) + 10·degraded_rate`), then pick the winner for production traffic.

This closes the loop: **agent pattern → routing policy → Opik trace → fitness → next policy**. The same machinery works against the Opik dashboard via its REST API; using JSONL keeps the demo dependency-free.

In [ ]:
import json

FITNESS_TRACE = './traces/tutorial_02_fitness.jsonl'

def score_policy(span_records, policy_axis):
    """Lower is better. Mean latency + 10s penalty per degraded span."""
    spans = [s for s in span_records
             if s.get('metadata', {}).get('routing_policy', {}).get('dominant_axis') == policy_axis]
    if not spans:
        return float('inf')
    elapsed = sum(s.get('metadata', {}).get('elapsed_seconds', 0) for s in spans) / len(spans)
    degraded_rate = sum(1 for s in spans if s.get('metadata', {}).get('degraded')) / len(spans)
    return elapsed + 10.0 * degraded_rate

if LIVE and s_patterns is not None:
    enable_tracing(backend='jsonl', output=FITNESS_TRACE)

    PROBE_QUESTIONS = [
        'Who is the CEO of Apple?',
        'Who is the CEO of NVIDIA?',
        'List all CEOs and their companies.',
    ]

    # Run each probe under each policy. Cheap because the questions are short.
    for axis_label, policy in [('latency               ', RoutingPolicy.fast()),
                                ('balanced              ', RoutingPolicy.balanced()),
                                ('information_quality   ', RoutingPolicy.thorough())]:
        cfg = AgentConfig(execution_mode='supervisor', handoff=True, routing_policy=policy)
        s_probe = Seocho.local(ontology, llm=LLM_SPEC, agent_config=cfg, user_id='openai-yitae')
        try:
            with s_probe.session(f'probe-{policy.dominant_axis}') as sess:
                for q in PROBE_QUESTIONS:
                    sess.run(q)
        finally:
            s_probe.close()
    flush_tracing()
    disable_tracing()

    # Replay the trace file and score each policy.
    spans = [json.loads(line) for line in open(FITNESS_TRACE) if line.strip()]
    for policy_axis in ('latency', 'balanced', 'information_quality'):
        score = score_policy(spans, policy_axis)
        print(f'  policy={policy_axis:<22}  fitness={score:.2f}  (lower is better)')

    winner = min(('latency', 'balanced', 'information_quality'),
                 key=lambda axis: score_policy(spans, axis))
    print(f'\n→ Recommended next-run policy axis: {winner}')
else:
    print('[SKIP]')

### 6.10 The closed loop

With §6.8 and §6.9 in place, **agent pattern, routing policy, and Opik flow are no longer three separate sections** — they're one self-reinforcing loop:

```
   pattern (run with policy X)
        ↓
   span emitted to Opik / JSONL with policy X metadata
        ↓
   downstream code reads the span, computes fitness for policy X
        ↓
   next call picks policy Y based on the fitness score
        ↓
   ... (loop)
```

The next section answers the question this loop sets up: **for a given workload, which mode should you start with?**

## 7. When to pick which mode

| You want… | Pick |
|-----------|------|
| Deterministic batch ingest, lowest cost, no retries | `pipeline` (default) |
| Single-task adaptivity (re-extract on low score, repair empty queries) | `agent` |
| Conversational front-door that decides whether to index or query | `supervisor` |
| Strict latency SLO | `supervisor_fast` or `agent` + `RoutingPolicy.fast()` |
| Highest-quality extraction, willing to spend tokens | `strict` or `supervisor_thorough` |

All three modes route through `extraction.agents_runtime.get_agents_runtime().run(...)`, so trace records, observability hooks, and degraded-state reporting work identically.

**Next:** Tutorial 3 brings in a real ontology (FIBO). Same agent, same multi-turn design, same Opik traces — but the system prompt and the tools' SHACL validation are now driven by a versioned schema with a `context_hash` you can check at every span.